In [1]:
import sys
original_sys_path = sys.path.copy()
sys.path.append('../')
from utils_models import *

dq.set_device('gpu')
import warnings
warnings.filterwarnings("ignore")

In [2]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_transmon = 4


EJ = 2
EC = EJ/4
EL = EJ/20
n_lvls_fluxonium = 20
n_lvls_resonator = 10
Er = 7
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": EJ, "Ec": EC, "El": EL, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )

res = MyResonator.create(
    n_lvls_resonator,
    params = {"ω":Er,"α":0},
    use_linear = False,
)

devices = [qsf, res]
f_indx = 0
r_indx = 1
Ns = [device.N for device in devices]
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
rn = qs.promote(res.ops['n'], r_indx, Ns)

g_tf = 0.2
system = qs.System.create(devices, couplings=[
    g_tf *  fn @ rn
    ])


# system.params["g_tf"] = g_tf
system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
driven_op = transform_op_into_dressed_basis_jax(rn, system_evecs_sorted.T)


system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]



In [3]:
system_scq = FluxoniumOscillatorSystem(
    EJ = EJ,
    EC = EC,
    EL = EL,
    Er = Er,
    g_strength = g_tf,
    qubit_level = n_lvls_fluxonium,
    osc_level = n_lvls_resonator,
    # products_to_keep=[[ql, ol] for ql in [0] for ol in range(50) ],
    computaional_states = '1,2',
    )

# hilbert spaace evals the same

In [4]:
np.allclose(system_scq.evals,system_evals_sorted )

True

# bare op the same

In [5]:
np.allclose(np.array(res.common_ops()['n'].data),np.array(system_scq.osc.n_operator()))

True

# bare hamiltonians the same

In [6]:
np.allclose(np.array(system.get_H_bare().data), np.array(system_scq.hilbertspace.bare_hamiltonian()))

True

# hilbertspace: interaction hamiltonians or full hamiltonians have ONLY sign differences

In [7]:
np.allclose(np.array(qutip.Qobj(np.array(system.get_H_couplings().data)).tidyup()),
             np.array(qutip.Qobj(np.array(system_scq.hilbertspace.interaction_hamiltonian())).tidyup())), \
    np.sum(np.abs(np.array(qutip.Qobj(np.array(system.get_H_couplings().data)).tidyup())) 
       - np.abs(np.array(qutip.Qobj(np.array(system_scq.hilbertspace.interaction_hamiltonian())).tidyup())) > 1e-6)

(False, 0)

In [8]:
np.allclose(np.array(qutip.Qobj(np.array(system.get_H().data)).tidyup()),
             np.array(qutip.Qobj(np.array(system_scq.hilbertspace.hamiltonian())).tidyup())), \
    np.sum(np.abs(np.array(qutip.Qobj(np.array(system.get_H().data)).tidyup())) 
       - np.abs(np.array(qutip.Qobj(np.array(system_scq.hilbertspace.hamiltonian())).tidyup())) > 1e-6)

(False, 0)

# Driven operator up to sign difference

In [9]:
np.allclose(np.abs(np.array(driven_op)), np.abs(system_scq.driven_operator.full()))

True